Here we wish to define the range of warpclasses to use.

This should consider both physics, redshift limit, number count etc.import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

We used this to define the add_warpclasses method. 

Might have to revisit this after looking at numbers after limiting to good data.

Shift definitions into the warp programs.

Also look at the existing sncosmo templates and determine what should be consder close matches. 

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
sys.path.append('/Users/jnordin/github/ampelFeb25')

In [ ]:
from warpTemplate import add_warpclasses, TEMPLATE_CLOSE_TYPES

In [ ]:
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')
print(df_bts.shape)

In [ ]:
df = add_warpclasses(df_bts, purge=True)

In [ ]:
df_bts['type'].value_counts()

In [ ]:
df['type'].value_counts()

In [ ]:
df['type_n'].value_counts()

In [ ]:
df['type_e'].value_counts()

In [ ]:
df['type_w'].value_counts()

In [ ]:
df['type_a'].value_counts()

In [ ]:
# require redshift and type as per the bts explorer
df_bts = df_bts.loc[ (df_bts['type'] != '-') & (df_bts['redshift'] != '-') ]
print(df_bts.shape)

In [ ]:
# We first remove non sn objects directly
nosn = ['AGN', 'Ca-rich', 'ILRT', 'LBV', 'LRN', 'Other', 'TDE', 'nova', 'other','SN Ca-rich-Ca']
df_bts = df_bts.loc[~df_bts['type'].isin(nosn)]
print(df_bts.shape)

In [ ]:
df_bts['type'].value_counts()

In [ ]:
list( set( df_bts['type'] ) )

In [ ]:
# We will never be able to do anything with less than 10 counts. So immediately we redefine these.
wmapI = {
    'SN II-pec':'SN II',
    'SN Ib-pec':'SN Ib',
    'SN Icn': 'SN Ic',
    'SN IIL': 'SN II',
    'SN IIn-pec': 'SN IIn',
    'SN Ic-pec': 'SN Ic',
}

In [ ]:
df_bts['type_n'] = df_bts['type'].replace( wmapI )

In [ ]:
# This will be the initial set of narrow channels
df_bts['type_n'].value_counts()

In [ ]:
wmap_tmp = {k:k+' (e)' for k in set(df_bts['type_n'])}

In [ ]:
wmap_tmp

In [ ]:
# We next decide on a next tier of extended classes 
wmapII = {
    'SN Ia-91bg': 'SN Ia-91bg (e)',
    'SN IIn': 'SN II (e)',
    'SN IIb': 'SN II (e)',
    'SN Ia-CSM': 'SN Ia (e)',
    'SN Ibn': 'SN Ib (e)',
    'SN Ia-SC': 'SN Ia (e)',
    'SN Ib': 'SN Ib (e)',
    'SLSN-II': 'SLSN (e)',
    'SN Iax': 'SN Iax (e)',
    'SN Ia-91T': 'SN Ia-91T (e)',
    'SLSN-I': 'SLSN (e)',
    'SN Ic': 'SN Ic (e)',
    'SN Ia-pec': 'SN Ia (e)',
    'SN IIP': 'SN II (e)',
    'SN Ic-BL': 'SN Ic (e)',
    'SN Ia': 'SN Ia (e)',
    'SN II': 'SN II (e)',
    'SN Ib/c': 'SN Ib/c (e)'
}

In [ ]:
df_bts['type_e'] = df_bts['type_n'].replace( wmapII )

In [ ]:
df_bts['type_e'].value_counts()

In [ ]:
wmap_tmp2 = {k:k+' (w)' for k in set(df_bts['type_e'])}

In [ ]:
wmap_tmp2

In [ ]:
# We next decide on a next tier of wide classes 
wmapIII = {
    'SN II (e)': 'SN II (w)',
    'SN Ib (e)': 'SN Ib/c (w)',
    'SN Ia-91T (e)': 'SN Ia (w)',
    'SLSN (e)': 'SLSN (w)',
    'SN Iax (e)': 'SN Iax (w)',
    'SN Ia-91bg (e)': 'SN Ia-91bg (w)',
    'SN Ic (e)': 'SN Ib/c (w)',
    'SN Ia (e)': 'SN Ia (w)',
    'SN Ib/c (e)': 'SN Ib/c (w)'
}

In [ ]:
df_bts['type_w'] = df_bts['type_e'].replace( wmapIII )

In [ ]:
df_bts['type_w'].value_counts()

In [ ]:
wmap_tmp3 = {k:k+' (a)' for k in set(df_bts['type_w'])}

In [ ]:
wmap_tmp3

In [ ]:
# Final set of ultra wide "all" classes
wmapIV = {
    'SN Ia (w)': 'SN Ia (a)',
    'SLSN (w)': 'SN CC (a)',
    'SN Ib/c (w)': 'SN CC (a)',
    'SN Iax (w)': 'SN Ia (a)',
    'SN II (w)': 'SN CC (a)',
    'SN Ia-91bg (w)': 'SN Ia (a)'
}

In [ ]:
df_bts['type_a'] = df_bts['type_w'].replace( wmapIV )

In [ ]:
df_bts['type_a'].value_counts()

In [ ]:
def add_warpclasses(df, classcolumn='type', purge=False):
    """
    Modify pandas dataframe according to the warp classes.
    """

    if purge:
        # Remove rows not in this list of classes:
        known_classes = [
            'SN IIn-pec',
             'SN II',
             'SN IIb',
             'SN Ia-CSM',
             'SN Ic-pec',
             'SN Ia-SC',
             'SN Iax',
             'SN Ia-pec',
             'SN IIP',
             'SN Icn',
             'SN Ib',
             'SN Ia',
             'SN Ia-91bg',
             'SN Ib-pec',
             'SN II-pec',
             'SLSN-I',
             'SN Ic-BL',
             'SN IIn',
             'SN Ic',
             'SN IIL',
             'SN Ib/c',
             'SLSN-II',
             'SN Ia-91T',
             'SN Ibn'
        ]
        df = df.loc[df[classcolumn].isin(known_classes)]

    # Step one - combine minor classes to base set of narrow types
    wmapI = {
        'SN II-pec':'SN II',
        'SN Ib-pec':'SN Ib',
        'SN Icn': 'SN Ic',
        'SN IIL': 'SN II',
        'SN IIn-pec': 'SN IIn',
        'SN Ic-pec': 'SN Ic',
    }    
    df[classcolumn+'_n'] = df[classcolumn].replace( wmapI )

    # Step two - combine narrow classes to extended
    wmapII = {
        'SN Ia-91bg': 'SN Ia-91bg (e)',
        'SN IIn': 'SN II (e)',
        'SN IIb': 'SN II (e)',
        'SN Ia-CSM': 'SN Ia (e)',
        'SN Ibn': 'SN Ib (e)',
        'SN Ia-SC': 'SN Ia (e)',
        'SN Ib': 'SN Ib (e)',
        'SLSN-II': 'SLSN (e)',
        'SN Iax': 'SN Iax (e)',
        'SN Ia-91T': 'SN Ia-91T (e)',
        'SLSN-I': 'SLSN (e)',
        'SN Ic': 'SN Ic (e)',
        'SN Ia-pec': 'SN Ia (e)',
        'SN IIP': 'SN II (e)',
        'SN Ic-BL': 'SN Ic (e)',
        'SN Ia': 'SN Ia (e)',
        'SN II': 'SN II (e)',
        'SN Ib/c': 'SN Ib/c (e)'
    }
    df[classcolumn+'_e'] = df[classcolumn+'_n'].replace( wmapII )

    # Step three - combine extended classes to wide 
    wmapIII = {
        'SN II (e)': 'SN II (w)',
        'SN Ib (e)': 'SN Ib/c (w)',
        'SN Ia-91T (e)': 'SN Ia (w)',
        'SLSN (e)': 'SLSN (w)',
        'SN Iax (e)': 'SN Iax (w)',
        'SN Ia-91bg (e)': 'SN Ia-91bg (w)',
        'SN Ic (e)': 'SN Ib/c (w)',
        'SN Ia (e)': 'SN Ia (w)',
        'SN Ib/c (e)': 'SN Ib/c (w)'
    }
    df[classcolumn+'_w'] = df[classcolumn+'_e'].replace( wmapIII )

    # Step for - combine to final all classes
    # Final set of ultra wide "all" classes
    wmapIV = {
        'SN Ia (w)': 'SN Ia (a)',
        'SLSN (w)': 'SN CC (a)',
        'SN Ib/c (w)': 'SN CC (a)',
        'SN Iax (w)': 'SN Ia (a)',
        'SN II (w)': 'SN CC (a)',
        'SN Ia-91bg (w)': 'SN Ia (a)'
    }
    df[classcolumn+'_a'] = df[classcolumn+'_w'].replace( wmapIV )

    return df

In [ ]:
df3 = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')

In [ ]:
df3 = add_warpclasses( df3, purge=True )

In [ ]:
set( df2['type_w'] )

Template classes 

In [ ]:
classfile = '/Users/jnordin/data/models/sncosmo/sncosmo_timeseriesmodels.csv'

In [ ]:
df_class = pd.read_csv(classfile,sep=';')

In [ ]:
TEMPLATE_CLOSE_TYPES = {
    'PopIII' : ['SLSN (w)'],
    'SN II' : [
        'SN II (n)', 'SN IIn (n)', 'SN IIb (n)',   # Should the IIb (n) class be here?
        'SN II (w)', 'SN CC (a)'
    ],
    'SN II-pec': ['SN II (w)', 'SN CC (a)'],
    'SN IIL': [
        'SN II (n)', 'SN IIn (n)', 'SN IIb (n)',   # Should the IIb (n) class be here?
        'SN II (w)', 'SN CC (a)'
    ],
    'SN IIL/P': [
        'SN II (n)', 'SN IIn (n)', 'SN IIb (n)',   # Should the IIb (n) class be here?
        'SN II (w)', 'SN CC (a)'
    ],
    'SN IIP': [
        'SN IIP (n)',
        'SN II (w)', 'SN CC (a)'
    ],
    'SN IIb': [
        'SN IIb (n)',
        'SN II (w)', 'SN CC (a)'
    ],
    'SN IIn': [
        'SN IIn (n)', 'SLSN (w)',   # SLSN ...  
        'SN II (w)', 'SN CC (a)',
    ],
    'SN Ia': [
        'SN Ia (n)', 'Ia-pec (n)', 'Ia-CSM (n)', 'Ia-SC (n)', 'Ia-91T (n)',     # Adding all the Ia subclasses since few other templates
        'SN Ia (w)', 'SN Ia (a)', 
        'SLSN (w)',
    ],
    'SN Ib': [
        'SN Ib (n)', 'SN Ibn (n)',     
        'SN Ib/c (w)', 'SN CC (a)'
    ],
    'SN Ib/c': [
        'SN Ib/c (w)', 'SN CC (a)'
    ],
    'SN Ic': [
        'SN Ic (n)',      
        'SN Ib/c (w)', 'SN CC (a)', 
    ],
    'SN Ic-BL': [
        'SN Ic (n)',      
        'SN Ib/c (w)', 'SN CC (a)',
    ],
}

In [ ]:
TEMPLATE_CLOSE_TYPES